In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import threading
import speech_recognition as sr
import pyttsx3
import time

# Inicialización de MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands()
mp_drawing = mp.solutions.drawing_utils

# Inicialización de reconocimiento de voz
r = sr.Recognizer()
mic = sr.Microphone()

# Inicialización de síntesis de voz
engine = pyttsx3.init()

# Variables globales
comando_actual = ""
color_actual = (255, 0, 0)  # Rojo
pos_x, pos_y = 300, 300

# Función para reconocer comandos de voz
def reconocer_comando():
    global comando_actual
    while True:
        with mic as source:
            print("🎤 Escuchando...")
            audio = r.listen(source)
        try:
            comando = r.recognize_google(audio, language="es-ES")
            comando_actual = comando.lower()
            print("👉 Comando:", comando_actual)
        except:
            comando_actual = ""
            print("❌ No entendí...")

# Hilo para voz
t = threading.Thread(target=reconocer_comando)
t.daemon = True
t.start()

# Captura de webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    # Procesar gestos
    mano_abierta = False
    dedos_arriba = 0

    if result.multi_hand_landmarks:
        for lm in result.multi_hand_landmarks[0].landmark[5:]:  # saltamos wrist
            if lm.y < result.multi_hand_landmarks[0].landmark[0].y:
                dedos_arriba += 1
        mano_abierta = dedos_arriba >= 4

        # Dibujar landmarks
        mp_drawing.draw_landmarks(frame, result.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)

    # Lógica condicional
    if "cambiar" in comando_actual and mano_abierta:
        color_actual = (0, 255, 0)  # Verde
        engine.say("¡Color cambiado a verde!")
        engine.runAndWait()
        comando_actual = ""  # para que no repita
    elif "mover" in comando_actual and dedos_arriba == 2:
        pos_x += 10
        engine.say("¡Moviendo objeto!")
        engine.runAndWait()
        comando_actual = ""

    # Visualización de rectángulo
    overlay = frame.copy()
    cv2.rectangle(overlay, (pos_x - 50, pos_y - 50), (pos_x + 50, pos_y + 50), color_actual, -1)
    alpha = 0.5
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

    # Mostrar información en pantalla
    cv2.putText(frame, f"Comando: {comando_actual}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(frame, f"Dedos arriba: {dedos_arriba}", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imshow("🎬 Escena Reactiva", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC para salir
        break

cap.release()
cv2.destroyAllWindows()